# Name That LLM: TF-IDF + style features

Generated by `python -m src.build_notebook` from
https://github.com/Dana-Lev/name-that-llm. The cells below recreate the `src/`
package, then fit on `train.csv` and write `submission.parquet`.

In [ ]:
from pathlib import Path

for folder in ['src', 'src/features', 'src/models']:
    Path(folder).mkdir(parents=True, exist_ok=True)
for empty_file in ['src/__init__.py', 'src/features/__init__.py']:
    Path(empty_file).touch()

In [ ]:
%%writefile src/data.py
"""Loading the competition data and building the cross-validation folds."""

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# The 18 valid labels, grouped by lab. Order is fixed: column k of every
# probability matrix in this project refers to LABELS[k].
LABS = {
    "OpenAI": ["gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra"],
    "Google": [
        "gemini-3.1-pro-preview",
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        "gemini-3.8-flash",
    ],
    "Anthropic": [
        "claude-fable-5",
        "claude-opus-4-6",
        "claude-opus-4-8",
        "claude-opus-5",
        "claude-sonnet-5",
    ],
    "Qwen": ["qwen3.8-max"],
    "ZAI": ["glm-5.3", "glm-5.3-flash"],
    "Moonshot AI": ["kimi-k3"],
}
LABELS = [model for models in LABS.values() for model in models]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
LAB_OF = {model: lab for lab, models in LABS.items() for model in models}

N_SPLITS = 5
SEED = 42


def find_data_dir():
    """The folder with train.csv: $NTL_DATA_DIR, Kaggle's input folder, or ./data."""
    candidates = [
        os.environ.get("NTL_DATA_DIR"),
        "/kaggle/input/competitions/name-that-llm",
        "/kaggle/input/name-that-llm",
        "data",
    ]
    for candidate in candidates:
        if candidate and (Path(candidate) / "train.csv").exists():
            return Path(candidate)
    raise FileNotFoundError("train.csv not found; set NTL_DATA_DIR or put the data in ./data")


def load_train(data_dir="data"):
    """Read train.csv and add an integer `label` column."""
    df = pd.read_csv(Path(data_dir) / "train.csv")
    unknown = set(df["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"train.csv has labels outside the 18 valid ones: {sorted(unknown)}")
    df["label"] = df["model"].map(LABEL_TO_ID)
    return df


def load_test(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "test.csv")


def make_folds(df, n_splits=N_SPLITS, seed=SEED):
    """Assign each row a fold id in [0, n_splits).

    All passages answering the same prompt land in the same fold, so every
    validation prompt is unseen during training, as in the hidden test set.
    Folds are also stratified by model to keep the class mix similar.
    """
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.full(len(df), -1, dtype=int)
    for fold, (_, valid_idx) in enumerate(
        splitter.split(df["text"], df["label"], groups=df["prompt_id"])
    ):
        folds[valid_idx] = fold
    return folds


In [ ]:
%%writefile src/features/tfidf.py
"""TF-IDF features over word and character n-grams."""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion


def build_tfidf(word_ngrams=(1, 2), char_ngrams=(2, 5), min_df=2):
    """Word n-grams capture vocabulary; character n-grams capture punctuation
    and formatting habits. Case is kept because capitalisation is a style cue."""
    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=word_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b|[^\w\s]",
    )
    char = TfidfVectorizer(
        analyzer="char",
        ngram_range=char_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
    )
    return FeatureUnion([("word", word), ("char", char)])


In [ ]:
%%writefile src/features/stylometry.py
"""Hand-made writing-style features: sentence shape, punctuation habits, formatting.

TF-IDF sees which words and character sequences appear. These features add what
it can't see: how long and how varied the sentences are, how the passage starts
and ends, and how often each punctuation mark is used per 100 characters.
"""

import re

import numpy as np

PUNCTUATION = {
    "comma": ",",
    "semicolon": ";",
    "colon": ":",
    "em_dash": "—",
    "en_dash": "–",
    "hyphen": "-",
    "open_paren": "(",
    "close_paren": ")",
    "straight_double_quote": '"',
    "straight_single_quote": "'",
    "curly_apostrophe": "’",
    "curly_open_quote": "“",
    "question": "?",
    "exclamation": "!",
    "ellipsis_char": "…",
    "three_dots": "...",
    "asterisk": "*",
    "backtick": "`",
    "hash": "#",
    "dollar": "$",
    "backslash": "\\",
    "slash": "/",
    "percent": "%",
    "equals": "=",
    "newline": "\n",
    "blank_line": "\n\n",
    "double_space": "  ",
}

SHAPE_NAMES = [
    "n_chars",
    "n_words",
    "n_sentences",
    "sentence_len_mean",
    "sentence_len_std",
    "sentence_len_max",
    "sentence_len_min",
    "word_len_mean",
    "word_len_std",
    "long_word_share",
    "distinct_word_share",
    "uppercase_share",
    "digit_share",
    "non_ascii_share",
    "starts_uppercase",
    "ends_with_sentence_mark",
    "ends_with_period",
    "pronoun_share",
    "n_list_items",
    "n_bold_spans",
    "n_latex_spans",
]
FEATURE_NAMES = SHAPE_NAMES + [f"{name}_per_100_chars" for name in PUNCTUATION]

_SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
_PRONOUN = re.compile(r"\b(?:I|we|you)\b")
_LIST_ITEM = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s", flags=re.M)
_BOLD = re.compile(r"\*\*[^*]+\*\*")
_LATEX = re.compile(r"\$[^$]+\$")


def style_vector(text):
    """The style features of one passage, in FEATURE_NAMES order."""
    words = text.split()
    sentences = [s for s in _SENTENCE_END.split(text) if s.strip()]
    sentence_lens = np.array([len(s.split()) for s in sentences] or [0])
    word_lens = np.array([len(w) for w in words] or [0])
    n_chars = max(len(text), 1)
    n_words = max(len(words), 1)
    last = text.rstrip()[-1:]

    shape = [
        len(text),
        len(words),
        len(sentences),
        sentence_lens.mean(),
        sentence_lens.std(),
        sentence_lens.max(),
        sentence_lens.min(),
        word_lens.mean(),
        word_lens.std(),
        (word_lens > 8).mean(),
        len({w.lower() for w in words}) / n_words,
        sum(c.isupper() for c in text) / n_chars,
        sum(c.isdigit() for c in text) / n_chars,
        sum(ord(c) > 127 for c in text) / n_chars,
        float(text[:1].isupper()),
        float(last in ".!?" and last != ""),
        float(last == "."),
        len(_PRONOUN.findall(text)) / n_words,
        len(_LIST_ITEM.findall(text)),
        len(_BOLD.findall(text)),
        len(_LATEX.findall(text)),
    ]
    punctuation = [text.count(mark) / n_chars * 100 for mark in PUNCTUATION.values()]
    return shape + punctuation


def style_features(texts):
    """Matrix of style features, one row per passage."""
    return np.array([style_vector(text) for text in texts], dtype=np.float64)


In [ ]:
%%writefile src/models/linear.py
"""Linear models: logistic regression on TF-IDF features and on style features."""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.features.stylometry import style_features
from src.features.tfidf import build_tfidf


def build_tfidf_logreg(C=10.0, max_iter=3000, **tfidf_kwargs):
    """class_weight="balanced" matches the metric, which weights every model equally."""
    return Pipeline(
        [
            ("tfidf", build_tfidf(**tfidf_kwargs)),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


def build_style_logreg(C=1.0, max_iter=3000):
    """The style features are on very different scales, so standardise them first."""
    return Pipeline(
        [
            ("style", FunctionTransformer(style_features)),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


In [ ]:
%%writefile src/models/blend.py
"""Combine several models by adding their log-probabilities."""

import numpy as np

EPS = 1e-12


class LogProbBlend:
    """Weighted sum of the members' log-probabilities, renormalised.

    Adding log-probabilities multiplies probabilities, so a class scores high
    only if every member finds it plausible. A weight below 1 makes a member's
    opinion count for less.
    """

    def __init__(self, members):
        """members: list of (build_model, weight) pairs."""
        self.members = members

    def fit(self, texts, y):
        self.models_ = [build().fit(texts, y) for build, _ in self.members]
        self.classes_ = self.models_[0].classes_
        return self

    def predict_proba(self, texts):
        total = sum(
            weight * np.log(model.predict_proba(texts) + EPS)
            for model, (_, weight) in zip(self.models_, self.members)
        )
        total -= total.max(axis=1, keepdims=True)
        probs = np.exp(total)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


In [ ]:
%%writefile src/models/__init__.py
"""Every model that can be cross-validated or submitted, by name."""

from src.models.blend import LogProbBlend
from src.models.linear import build_style_logreg, build_tfidf_logreg


def build_tfidf_style_blend(style_weight=0.5):
    return LogProbBlend([(build_tfidf_logreg, 1.0), (build_style_logreg, style_weight)])


MODELS = {
    "tfidf_logreg": build_tfidf_logreg,
    "style_logreg": build_style_logreg,
    "tfidf_style_blend": build_tfidf_style_blend,
}


In [ ]:
%%writefile src/prompts.py
"""Using the structure of the data: every model answers each prompt at most once.

In train.csv no prompt has two passages from the same model. If the test set
is built the same way, the predictions for one prompt should all be different
models. The test set has no prompt_id, so we first group passages by topic
(passages answering one prompt share vocabulary), then pick the best labelling
of each group in which no model repeats.

Whether the hidden test set follows this structure is an assumption; the
leaderboard is the only way to check it.
"""

import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer

EPS = 1e-12


def cluster_prompts(texts, threshold=0.90):
    """Group passages that seem to answer the same prompt; returns a group id per passage.

    Average-linkage clustering on cosine distance between TF-IDF vectors with
    stop words removed, so only topic words count. Merging stops at `threshold`,
    so the number of prompts doesn't have to be known. 0.90 was chosen on
    train.csv, where the true prompts are known.
    """
    texts = list(texts)
    if len(texts) < 2:
        return np.zeros(len(texts), dtype=int)
    vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=1)
    vectors = vectorizer.fit_transform(texts).toarray()
    # A passage with no topic words has a zero vector, which has no cosine distance.
    empty = ~vectors.any(axis=1)
    vectors[empty, 0] = 1.0
    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold, metric="cosine", linkage="average"
    )
    return clustering.fit_predict(vectors)


def assign_within_groups(probs, groups):
    """Label each group so that no class repeats, maximising total log-probability.

    This is the assignment problem, solved exactly by the Hungarian algorithm.
    A group larger than the number of classes can't be a single prompt (the
    clustering merged several), so there each class may be used once per
    merged prompt: ceil(group size / number of classes) times.
    """
    probs = np.asarray(probs)
    groups = np.asarray(groups)
    n_classes = probs.shape[1]
    log_probs = np.log(probs + EPS)
    predicted = np.zeros(len(probs), dtype=int)
    for group in np.unique(groups):
        rows = np.where(groups == group)[0]
        copies = int(np.ceil(len(rows) / n_classes))
        row_idx, col_idx = linear_sum_assignment(-np.tile(log_probs[rows], (1, copies)))
        predicted[rows[row_idx]] = col_idx % n_classes
    return predicted


In [ ]:
%%writefile src/predict.py
"""Fit a model on all of train.csv and write submission.parquet.

    python -m src.predict --model tfidf_style_blend --data-dir data

This is what the Kaggle submission notebooks run.
"""

import argparse

import pandas as pd

from src.data import LABELS, find_data_dir, load_test, load_train
from src.models import MODELS
from src.prompts import assign_within_groups, cluster_prompts


def check_submission(submission, test):
    """Fail here rather than have Kaggle reject the file."""
    if list(submission.columns) != ["id", "model"]:
        raise ValueError(f"columns must be id, model; got {list(submission.columns)}")
    if submission.isna().any().any():
        raise ValueError("submission has null values")
    if submission["id"].duplicated().any() or set(submission["id"]) != set(test["id"]):
        raise ValueError("every test id must appear exactly once")
    unknown = set(submission["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"labels outside the 18 valid ones: {sorted(unknown)}")


def make_submission(model="tfidf_logreg", assign=False, data_dir=None, out="submission.parquet"):
    """With assign=True, group the test passages by prompt and let no model
    repeat inside a group (see src/prompts.py)."""
    data_dir = find_data_dir() if data_dir is None else data_dir
    train, test = load_train(data_dir), load_test(data_dir)

    fitted = MODELS[model]().fit(train["text"], train["label"])
    # Columns of predict_proba follow fitted.classes_, which are label ids.
    probs = fitted.predict_proba(test["text"])
    if assign:
        groups = cluster_prompts(test["text"])
        columns = assign_within_groups(probs, groups)
        print(f"grouped {len(test)} passages into {groups.max() + 1} prompts")
    else:
        columns = probs.argmax(axis=1)
    predicted = fitted.classes_[columns]

    submission = pd.DataFrame({"id": test["id"], "model": [LABELS[i] for i in predicted]})
    check_submission(submission, test)
    submission.to_parquet(out, index=False)
    print(f"trained on {len(train)} passages, wrote {len(submission)} predictions to {out}")
    return submission


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--model", default="tfidf_logreg", choices=sorted(MODELS))
    parser.add_argument("--assign", action="store_true")
    parser.add_argument("--data-dir", default=None)
    parser.add_argument("--out", default="submission.parquet")
    args = parser.parse_args()
    make_submission(args.model, args.assign, args.data_dir, args.out)


In [ ]:
from src.predict import make_submission

submission = make_submission(model='tfidf_style_blend', assign=False)
submission["model"].value_counts()